# 02b — A global flood model on new floods
**"I just want to use a model."** In notebook 02a we trained a small regional model. Now we use the model
trained on the whole WorldFloods v2 dataset (~500 flood maps from all over the world; Portalés-Julià et al.,
2023, *Scientific Reports*) on floods it has never seen, far from Greece.

1. Download the trained weights from HuggingFace.
2. Run the model on three new floods → cloud-aware flood maps (land / water / cloud).
3. Choose the decision threshold.
4. From a raster to polygons (GeoJSON) that a GIS or an emergency service can use.

**Google Colab:** run the next cell first (~1 min). It downloads the course code, data and models and installs
ml4floods; if Colab disconnects, just run it again. On your own computer it does nothing.

In [ ]:
import sys
if "google.colab" in sys.modules:
    import subprocess
    from pathlib import Path
    COURSE = Path("/content/geomundus-ml4floods")
    if not COURSE.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/kipoju/geomundus-ml4floods", str(COURSE)], check=True)
    sys.path.insert(0, str(COURSE / "notebooks"))
    import colab_setup
    colab_setup.setup()

In [ ]:
import sys
sys.path.append(".")

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from georeader import plot
from georeader.geotensor import GeoTensor
from georeader.save import save_cog
from georeader.vectorize import get_polygons
from huggingface_hub import hf_hub_download
from ml4floods.models.model_setup import get_channel_configuration_bands
from ml4floods.scripts.inference import load_inference_function
from course_utils import (BANDS_S2, ROOT, binary_metrics, composite, confusion_legend, confusion_overlay, find_event, interact,
                          read_event, save_fig)

## 1. Download the trained model
The ml4floods models live in the HuggingFace repository
[isp-uv-es/ml4floods](https://huggingface.co/isp-uv-es/ml4floods). Each model is a folder with a
`config.json` (architecture, input bands, normalisation…) and the weights `model.pt` — the same layout as the
model we saved in notebook 02a.

`WF2_unetv2_bgriswirs`: a U-Net trained on WorldFloods v2 with 6 bands (Blue, Green, Red, NIR, SWIR1,
SWIR2), which also exist in Landsat 8/9 → the same model works on both satellites.

In [ ]:
EXPERIMENT = "WF2_unetv2_bgriswirs"
for filename in ("config.json", "model.pt"):
    hf_hub_download(repo_id="isp-uv-es/ml4floods", subfolder=f"models/{EXPERIMENT}", filename=filename,
                    local_dir=ROOT)   # -> ROOT/models/WF2_unetv2_bgriswirs/ (skipped if already there)

# th_water: probability above which a pixel is water. This model was trained to favour recall (water pixels
# weigh 6× more in the loss) → with 0.5 it raises many false alarms; 0.7 gives a better balance (section 3).
# max_tile_size: the image is processed in tiles of at most 1024×1024 px to limit memory (512 or 256 if you
# run out of RAM).
TH_WATER = 0.7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # the GPU if there is one
predict, config = load_inference_function(str(ROOT / "models" / EXPERIMENT), device_name=DEVICE,
                                          max_tile_size=1024, th_water=TH_WATER)
channels = get_channel_configuration_bands(config.data_params.channel_configuration, collection_name="S2")
print("model input bands:", config.data_params.channel_configuration, channels)


def wf2_numpy(s2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Flood map {0: invalid, 1: land, 2: water, 3: cloud} and P(water) of the global model for a 13-band
    Sentinel-2 image (band, y, x). Same pattern as `unet_numpy` in 02a: GeoTensor in → GeoTensors out."""
    pred_class, pred_prob = predict(torch.tensor(np.nan_to_num(np.asarray(s2)[channels])))
    pred_class, prob_water = pred_class.numpy(), pred_prob[1].numpy()   # P(water) = -1 where the input is invalid
    if isinstance(s2, GeoTensor):
        pred_class, prob_water = s2.array_as_geotensor(pred_class, fill_value_default=0), s2.array_as_geotensor(prob_water)
    return pred_class, prob_water

## 2. Three floods the model has never seen
From the official WorldFloods v2 **test** set (never used to train or select the model):
- **Ylitornio, Finland** (EMSR9284, May 2018): ice-jam flood of the Torne river in a boreal landscape.
- **Miandrivazo, Madagascar** (EMSR264, Jan 2018): flash flood, sediment-laden rivers.
- **Portopalo, Sicily** (EMSR333, Nov 2018): flash flood under a partly cloudy sky.

The model outputs two probabilities per pixel, **P(cloud)** and **P(water)** (the *clouds-aware multioutput*
model of the slides). A simple rule turns them into a map: cloud if P(cloud) > 0.5 and the pixel is bright;
else water if P(water) > `TH_WATER`; else land.

In [ ]:
NEW_FLOODS = {"Ylitornio (Finland)": "EMSR9284_01YLITORNIONORTHERN_DEL_MONIT01_v1",
              "Miandrivazo (Madagascar)": "EMSR264_18MIANDRIVAZODETAIL_DEL_v2",
              "Portopalo (Sicily)": "EMSR333_02PORTOPALO_DEL_MONIT01_v1"}
COLORS_PRED = np.array([[0, 0, 0],          # 0: invalid
                        [139, 64, 0],       # 1: land
                        [0, 0, 240],        # 2: water
                        [220, 220, 220]],   # 3: cloud
                       dtype=np.float32) / 255

results = {}
for name, event in NEW_FLOODS.items():
    d = read_event(*find_event(event), event)
    pred_class, prob_water = wf2_numpy(d["s2"])
    results[name] = dict(s2=d["s2"], gt=d["gt"], pred_class=pred_class, prob_water=prob_water)
    print(name, "done")

In [ ]:
# One column per flood: Sentinel-2 (SWIR / NIR / Red) on top, the model's flood map below
fig, axs = plt.subplots(2, len(results), figsize=(6.2 * len(results), 11.5))
for col, (name, r) in zip(axs.T, results.items()):
    s2, gt = r["s2"], r["gt"]
    m = binary_metrics(r["pred_class"] == 2, gt[1] == 2, (gt[1] != 0) & (gt[0] != 2))
    swir = s2[[BANDS_S2.index(b) for b in ("B11", "B8", "B4")]]   # GeoTensor (3, y, x)
    plot.show(np.clip(swir / 4500, 0, 1), ax=col[0], add_scalebar=True)
    col[0].set_title(f"{name}\nSentinel-2 (B11, B8, B4)", fontsize=14)
    plot.plot_segmentation_mask(r["pred_class"], COLORS_PRED, ax=col[1],
                                interpretation_array=["invalid", "land", "water", "cloud"])
    col[1].set_title(f"Flood map, P(water) > {TH_WATER}\nvs reference: IoU {m['IoU']:.2f}", fontsize=14)
plt.tight_layout()
save_fig(fig, "dl_wf2_new_floods")

In [ ]:
# The raw model output: probability of water (before choosing a threshold)
fig, axs = plt.subplots(1, len(results), figsize=(6.2 * len(results), 5.8))
for ax, (name, r) in zip(axs, results.items()):
    im = ax.imshow(np.where(r["prob_water"] < 0, np.nan, r["prob_water"].values), cmap="Blues", vmin=0, vmax=1)
    cax = ax.inset_axes([0.05, 0.04, 0.4, 0.035])   # colorbar inside the panel: all panels keep their size
    fig.colorbar(im, cax=cax, orientation="horizontal", label="P(water)")
    ax.set_title(f"{name}: P(water)")
    ax.set_axis_off()
plt.tight_layout()
plt.show()

How good are these maps? The reference flood maps of WorldFloods (made by Copernicus EMS analysts) let us
check them, on the valid pixels outside the clouds of the reference:

In [ ]:
for name, r in results.items():
    gt = r["gt"]
    m = binary_metrics(r["pred_class"] == 2, gt[1] == 2, (gt[1] != 0) & (gt[0] != 2))
    print(f"{name:26s} P {m['precision']:.2f} · R {m['recall']:.2f} · IoU {m['IoU']:.2f}")

## 3. Choosing the decision threshold
The map depends on the threshold on P(water). Low threshold → we miss less water (**recall ↑**) but raise
more false alarms (**precision ↓**).

❓ An emergency service planning rescues: do they prefer high recall or high precision? And an insurance
company paying for damages?

In [ ]:
SLIDER_FLOOD = "Miandrivazo (Madagascar)"
r = results[SLIDER_FLOOD]
ref, valid = r["gt"][1] == 2, (r["gt"][1] != 0) & (r["gt"][0] != 2) & (r["prob_water"] >= 0)


def show_threshold(th=TH_WATER):
    p = r["prob_water"] > th
    m = binary_metrics(p, ref, valid)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(confusion_overlay(p, ref, valid))
    ax.set_title(f"{SLIDER_FLOOD}: P(water) > {th:.2f} — P {m['precision']:.2f} · R {m['recall']:.2f} · "
                 f"IoU {m['IoU']:.2f}")
    ax.set_axis_off()
    confusion_legend(ax)
    plt.show()


interact(show_threshold, th=widgets.FloatSlider(value=TH_WATER, min=0.05, max=0.95, step=0.05))

## 4. From raster to polygons
Emergency services work with vector layers (GIS, web maps). We turn the water pixels into polygons with
georeader and save them as GeoJSON: open the file in QGIS or on geojson.io.

In [ ]:
POLY_FLOOD = "Miandrivazo (Madagascar)"
r = results[POLY_FLOOD]
s2 = r["s2"]
out_dir = ROOT / "outputs"
out_dir.mkdir(exist_ok=True)
water = (r["pred_class"] == 2).astype(np.uint8)   # GeoTensor: pixels → map coordinates come for free
save_cog(water, str(out_dir / f"{NEW_FLOODS[POLY_FLOOD]}_{EXPERIMENT}_water.tif"), descriptions=["water"])   # raster
polygons = get_polygons(water, min_area=25)   # in pixels: drop polygons smaller than 25 px (0.25 ha)
flood = gpd.GeoDataFrame(geometry=polygons, crs=s2.crs)
print(f"{len(flood)} water polygons, total area {flood.area.sum() / 1e6:.1f} km²")

out_file = out_dir / f"{NEW_FLOODS[POLY_FLOOD]}_{EXPERIMENT}.geojson"
flood.to_crs("EPSG:4326").to_file(out_file, driver="GeoJSON")
print("saved", out_file)

fig, ax = plt.subplots(figsize=(10, 9))
plot.show(np.clip(s2[[BANDS_S2.index(b) for b in ("B4", "B3", "B2")]] / 3000, 0, 1), ax=ax, add_scalebar=True)
flood.boundary.plot(ax=ax, color="cyan", linewidth=0.6)
ax.set_title(f"{POLY_FLOOD}: {len(flood)} water polygons ({flood.area.sum() / 1e6:.1f} km²) → GeoJSON")
save_fig(fig, "dl_polygons")

**Take-aways**
- A model trained on many floods worldwide works on floods far from anything we trained on today, with one
  setting to choose: the threshold on P(water).
- It still has limits: dense clouds hide the water (optical sensor) → SAR or the next clear image.
- In a few lines: weights from HuggingFace → Sentinel-2 image → flood map → polygons for a GIS.
  On a new flood you only need a new Sentinel-2 (or Landsat) image of the area.